# CS604 Colab launcher (Ana): cached-feature linear probe

Runtime: **GPU** (A100 40GB, High-RAM off is enough). Same repo / Drive / data setup as `colab_launcher.ipynb`; the run step is `scripts/linear_probe_cached.py` instead of the trainer.

What it does: embeds every image once at `IMG_SIZE` (frozen ViT), sweeps `lr_sweep` on a 5-per-class holdout, retrains the head on all train images with the best lr, then writes `val_team_{TEAM}.txt` (+ `.zip`) and `probe_head.pt` to a run folder in `CS604_RUNS`.

The GitHub repo is private, so the clone cell will ask for a Personal Access Token (hidden). Do not paste the token into the notebook text or commit it.

Change `IMG_SIZE`, `TEAM`, and `BRANCH` below before you run. Other knobs (epochs, lrs, weight decay, …) live in `CONFIG`.

In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
FEATURE_DIR = "/content/features"  # local disk only, to save Drive space
BRANCH = "ana-branch"
CONFIG = "configs/linear_probe_cached.yaml"
IMG_SIZE = 224  # 224 is native; 336 only if the prof allows higher resolution
TEAM = "08"


## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Download data to `/content/data`

Skipped if the data is already unpacked in this session. If the download fails with 401, run the optional login cell first.

In [ ]:
# Optional: only if the dataset download asks for a Hugging Face login
# from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
from pathlib import Path

if (Path(DATA_DIR) / "train.txt").exists():
    print("Data already in", DATA_DIR)
else:
    !python scripts/download_data.py --dest {DATA_DIR}

## Review settings (table)

`CONFIG` merged with `base.yaml`, plus this notebook's overrides. This is exactly what the script will run with.

In [ ]:
import pandas as pd

from src.utils import apply_overrides, load_config

chosen = load_config(CONFIG)
apply_overrides(chosen, data_root=DATA_DIR, output_dir=DRIVE_OUT, feature_dir=FEATURE_DIR, img_size=IMG_SIZE)
display(pd.DataFrame(list(chosen.to_dict().items()), columns=["parameter", "value"]))

## 5. Probe: features → lr sweep → retrain → submission

Feature extraction takes roughly 5–15 min on an A100 at 224 (longer at 336); features are cached in `FEATURE_DIR` per backbone and size, so rerunning in the same session skips it.

In [ ]:
!python scripts/linear_probe_cached.py --config {CONFIG} --team {TEAM} --img_size {IMG_SIZE} \
    --data_root {DATA_DIR} --output_dir {DRIVE_OUT} --feature_dir {FEATURE_DIR}

## 6. Results

Loss curves: both falling together = still learning; train falling while holdout flattens = overfitting. With label smoothing 0.1 over 10,000 classes the loss floor is about 1.25, not 0.

In [ ]:
import json

from IPython.display import Image

run_dir = sorted(Path(DRIVE_OUT).glob(f"linear_probe_cached_{IMG_SIZE}_*"))[-1]
print("Run folder:", run_dir)
display(pd.read_csv(run_dir / "probe_sweep.csv"))
display(Image(filename=str(run_dir / "probe_curves.png")))
print(json.dumps(json.loads((run_dir / "probe_summary.json").read_text()), indent=2))
print("Upload to CodaBench:", run_dir / f"val_team_{TEAM}.zip")
print("Then add a row to report/experiment_index.csv and a note in report/progress_log.md")

## 7. (Optional) Shut down the runtime

In [ ]:
from google.colab import runtime

runtime.unassign()